In [2]:
import sys
import os

from xgboost import XGBClassifier

sys.path.append(os.path.abspath('..'))

from src.cv import *

X, y, X_test, test_ids = load_data(data_dir="../data")
X = prepare_categoricals(X, flavour="xgb")

_ = quick_cv(lambda: XGBClassifier(), X, y)

  fold 1/2  AUC=0.95744  (3s)  best_iter=99
  fold 2/2  AUC=0.95785  (2s)  best_iter=99
model
  fold AUCs   : 0.95744  0.95785
  mean +/- std: 0.95765 +/- 0.00020
  OOF AUC     : 0.95764   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 4.8s
  best_iters  : [99, 99]


In [13]:
"""
Try combinations of features
"""

X_temp = X.copy()

features_to_try = [
    ("total_screen", X["daily_screen_time_hours"] + X["weekend_screen_time"]),
    ("d_plus_2s", X["daily_screen_time_hours"] + 2 * X["social_media_hours"]),
    ("avg_screen", (5 * X["daily_screen_time_hours"] + 2 * X["weekend_screen_time"]) / 7),

    ("social_media_ratio", X["social_media_hours"] / (X["daily_screen_time_hours"] + 0.01)),
    ("portion_of_day_on_screen", X["daily_screen_time_hours"] / (24 - X["sleep_hours"])),
    ("screen_to_work_ratio", X["daily_screen_time_hours"] / (X["work_study_hours"] + 0.01)),
    ("leisure_ratio", (X["daily_screen_time_hours"] - X["work_study_hours"] - X["gaming_hours"]) / (X["daily_screen_time_hours"] + 0.01)),
    ("screen_share_of_free", (X["daily_screen_time_hours"] / (24 - X["sleep_hours"] - X["work_study_hours"]))),

    ("mins_per_open", X["daily_screen_time_hours"] * 60 / X["app_opens_per_day"]),
    ("notif_per_screen_hour", X["notifications_per_day"] / (X["daily_screen_time_hours"] + 0.01))
]

for name, col in features_to_try:
    X_temp[name] = col
    print(f"----- {name} -------")
    quick_cv(lambda: XGBClassifier(), X_temp, y)
    print("----------------------")
    X_temp = X.copy()


----- total_screen -------
  fold 1/2  AUC=0.95692  (8s)  best_iter=99
  fold 2/2  AUC=0.95793  (4s)  best_iter=99
model
  fold AUCs   : 0.95692  0.95793
  mean +/- std: 0.95743 +/- 0.00051
  OOF AUC     : 0.95743   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 12.5s
  best_iters  : [99, 99]
----------------------
----- d_plus_2s -------
  fold 1/2  AUC=0.95674  (6s)  best_iter=99
  fold 2/2  AUC=0.95791  (5s)  best_iter=99
model
  fold AUCs   : 0.95674  0.95791
  mean +/- std: 0.95733 +/- 0.00058
  OOF AUC     : 0.95733   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 11.9s
  best_iters  : [99, 99]
----------------------
----- avg_screen -------
  fold 1/2  AUC=0.95752  (2s)  best_iter=99
  fold 2/2  AUC=0.95752  (5s)  best_iter=99
model
  fold AUCs   : 0.95752  0.95752
  mean +/- std: 0.95752 +/- 0.00000
  OOF AUC     : 0.95752   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 7.2s
  best_

In [5]:
"""
Imputed columns: append 9 iteratively-imputed columns, keep the raw NaNs.
IterativeImputer learns from data, so it must be fitted inside the fold.
"""
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


class AddImputed:
    def fit_transform(self, X, y=None):
        self.imp = IterativeImputer(random_state=SEED).fit(X[NUM_COLS])
        return self.transform(X)

    def transform(self, X):
        X = X.copy()
        X[[c + "_imp" for c in NUM_COLS]] = self.imp.transform(X[NUM_COLS])
        return X

_ = quick_cv(lambda: XGBClassifier(), X, y, preprocessor=AddImputed)

/home/lukasbrookfield/.local/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  fold 1/2  AUC=0.95826  (14s)  best_iter=99


/home/lukasbrookfield/.local/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  fold 2/2  AUC=0.95821  (10s)  best_iter=99
model
  fold AUCs   : 0.95826  0.95821
  mean +/- std: 0.95824 +/- 0.00003
  OOF AUC     : 0.95824   <-- compare experiments on THIS number
  rows used   : 200,000
  time        : 23.9s
  best_iters  : [99, 99]


In [3]:
"""
Does adding the original dataset help?
Original rows go into the TRAINING folds only, so validation stays 100%
competition data and the AUC is comparable to every other run in results.md.
Same folds for every variant -> the per-fold difference is a paired comparison.
"""
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

SAMPLE, FOLDS, NOISE = 200_000, 2, 0.00035

X_orig = pd.read_csv("../data/train_original.csv")
y_orig = X_orig.pop("addicted_label")
X_orig = X_orig[X.columns].astype(X.dtypes)

Xs, _, ys, _ = train_test_split(X, y, train_size=SAMPLE, random_state=SEED, stratify=y)
Xs, ys = Xs.reset_index(drop=True), ys.reset_index(drop=True)

def variants(X_tr, y_tr):
    y_aug = pd.concat([y_tr, y_orig], ignore_index=True)
    return {
        "baseline": (X_tr, y_tr),
        "+original": (pd.concat([X_tr, X_orig], ignore_index=True), y_aug),
        "+original+flag": (pd.concat([X_tr.assign(is_original=0),
                                      X_orig.assign(is_original=1)], ignore_index=True), y_aug),
    }

scores = {k: [] for k in variants(Xs, ys)}
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (tr, va) in enumerate(skf.split(Xs, ys)):
    if fold >= FOLDS:
        break
    X_va, y_va = Xs.iloc[va], ys.iloc[va]
    for name, (X_tr, y_tr) in variants(Xs.iloc[tr], ys.iloc[tr]).items():
        X_eval = X_va.assign(is_original=0) if "flag" in name else X_va
        model = XGBClassifier(eval_metric="auc").fit(X_tr, y_tr)
        scores[name].append(roc_auc_score(y_va, model.predict_proba(X_eval)[:, 1]))
        print(f"  fold {fold + 1}  {name:15s} AUC={scores[name][-1]:.5f}")

base = np.array(scores["baseline"])
print(f"\nbaseline           AUC={base.mean():.5f}  per-fold={np.round(base, 5)}")

for name in ["+original", "+original+flag"]:
    d = np.array(scores[name]) - base
    if not ((d > 0).all() or (d < 0).all()):
        verdict = "REJECT - folds disagree on the sign, this is noise"
    elif d.mean() > NOISE:
        verdict = "WORTH PURSUING - confirm with a full run_cv"
    elif d.mean() < -NOISE:
        verdict = "REJECT - consistently hurts"
    else:
        verdict = "REJECT - smaller than fold noise"
    print(f"{name:18s} AUC={np.mean(scores[name]):.5f}  delta={d.mean():+.5f}  "
          f"per-fold={np.round(d, 5)}")
    print(f"  vs noise +/-{NOISE}  ->  {verdict}")


  fold 1  baseline        AUC=0.95744
  fold 1  +original       AUC=0.95728
  fold 1  +original+flag  AUC=0.95677
  fold 2  baseline        AUC=0.95785
  fold 2  +original       AUC=0.95788
  fold 2  +original+flag  AUC=0.95741

baseline           AUC=0.95765  per-fold=[0.95744 0.95785]
+original          AUC=0.95758  delta=-0.00006  per-fold=[-1.6e-04  3.0e-05]
  vs noise +/-0.00035  ->  REJECT - folds disagree on the sign, this is noise
+original+flag     AUC=0.95709  delta=-0.00055  per-fold=[-0.00067 -0.00044]
  vs noise +/-0.00035  ->  REJECT - consistently hurts


In [6]:
"""
Confirm the imputer lead on the FULL data: 5 folds, all 691k rows, identical
folds in both runs. n_estimators raised so early stopping, not the tree cap,
decides when to stop.
"""
from scipy import stats

make_xgb = lambda: XGBClassifier(n_estimators=1000, learning_rate=0.1, eval_metric="auc")

base = run_cv(make_xgb, X, y, name="xgb_full_base")
imp = run_cv(make_xgb, X, y, name="xgb_full_imputed", preprocessor=AddImputed)

d = np.array(imp.fold_scores) - np.array(base.fold_scores)
t, p = stats.ttest_rel(imp.fold_scores, base.fold_scores)

print(f"\nbaseline OOF AUC {base.oof_auc:.5f}")
print(f"imputed  OOF AUC {imp.oof_auc:.5f}   delta {imp.oof_auc - base.oof_auc:+.5f}")
print(f"per-fold delta   {np.round(d, 5)}")
print(f"mean {d.mean():+.5f} +/- {d.std(ddof=1):.5f} (std over folds)")
print(f"paired t-test    t={t:.2f}  p={p:.4f}")

if p < 0.05 and (d > 0).all():
    print("SIGNIFICANT - keep the imputed columns, try them on CatBoost next")
else:
    print("NOT SIGNIFICANT - within fold noise, drop it")


  fold 1/5  AUC=0.96318  (28s)  best_iter=997
  fold 2/5  AUC=0.96393  (30s)  best_iter=994
  fold 3/5  AUC=0.96415  (30s)  best_iter=999
  fold 4/5  AUC=0.96468  (29s)  best_iter=999
  fold 5/5  AUC=0.96363  (32s)  best_iter=996
xgb_full_base
  fold AUCs   : 0.96318  0.96393  0.96415  0.96468  0.96363
  mean +/- std: 0.96392 +/- 0.00050
  OOF AUC     : 0.96391   <-- compare experiments on THIS number
  rows used   : 691,369
  time        : 148.9s
  best_iters  : [997, 994, 999, 999, 996]


/home/lukasbrookfield/.local/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  fold 1/5  AUC=0.96396  (65s)  best_iter=982


/home/lukasbrookfield/.local/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  fold 2/5  AUC=0.96445  (72s)  best_iter=983


...

  fold 3/5  AUC=0.96467  (63s)  best_iter=964


/home/lukasbrookfield/.local/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  fold 4/5  AUC=0.96528  (66s)  best_iter=999


/home/lukasbrookfield/.local/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


  fold 5/5  AUC=0.96446  (53s)  best_iter=993
xgb_full_imputed
  fold AUCs   : 0.96396  0.96445  0.96467  0.96528  0.96446
  mean +/- std: 0.96456 +/- 0.00043
  OOF AUC     : 0.96456   <-- compare experiments on THIS number
  rows used   : 691,369
  time        : 319.2s
  best_iters  : [982, 983, 964, 999, 993]

baseline OOF AUC 0.96391
imputed  OOF AUC 0.96456   delta +0.00065
per-fold delta   [0.00077 0.00051 0.00052 0.0006  0.00083]
mean +0.00065 +/- 0.00015 (std over folds)
paired t-test    t=9.82  p=0.0006
SIGNIFICANT - keep the imputed columns, try them on CatBoost next
